# Vacancy formation energy in bcc Fe (EAM)

First refines the bcc Fe lattice parameter via an EOS scan
(`optimise_cubic_lattice_parameter`) starting from `a=2.85`, then
builds a supercell from the equilibrium structure, removes one
atom, and computes the vacancy formation energy
`E_vac - ((N-1)/N) * E_bulk` under the included Al-Fe.eam.fs EAM
potential.

Driven from a `Workflow` so that `engine.with_working_directory(...)`
is evaluated eagerly with the resolved engine value.


In [1]:
import pathlib
from ase.build import bulk
from ase.calculators.eam import EAM

from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.engine import ASEEngine, CalcInputMinimize, calculate
from pyiron_workflow_atomistics.structure import (
    create_supercell_with_min_dimensions,
    create_vacancy,
)
from pyiron_workflow_atomistics.physics.bulk import optimise_cubic_lattice_parameter
from pyiron_workflow_atomistics.physics.point_defect import (
    calculate_vacancy_formation_energy,
)

engine = ASEEngine(
    EngineInput=CalcInputMinimize(
        force_convergence_tolerance=0.05, max_iterations=100
    ),
    calculator=EAM(potential=str(next(p for p in [pathlib.Path("Al-Fe.eam.fs"), pathlib.Path("notebooks/Al-Fe.eam.fs"), pathlib.Path(__file__).parent / "Al-Fe.eam.fs" if "__file__" in globals() else pathlib.Path(".")] if p.exists()))),
    working_directory="./_vac_runs",
)


In [2]:
# Pre-build a supercell from the *initial* a=2.85 structure just to
# count atoms — N for E_f = E_vac - ((N-1)/N) * E_bulk must be a
# scalar at workflow-construction time. The optimised-a0 supercell
# uses the same min_dimensions, so it has the same atom count.
fe_supercell_init = create_supercell_with_min_dimensions.node_function(
    bulk("Fe", a=2.85, cubic=True), min_dimensions=[7, 7, 7]
)
n_atoms_supercell = len(fe_supercell_init)
print(f"supercell: {n_atoms_supercell} Fe atoms")

wf = Workflow("vacancy_formation_energy")
wf.lat_opt = optimise_cubic_lattice_parameter(
    structure=bulk("Fe", a=2.85, cubic=True),
    name="Fe",
    crystalstructure="bcc",
    engine=engine.with_working_directory("lat_opt"),
    strain_range=(-0.02, 0.02),
    num_points=5,
)
wf.supercell = create_supercell_with_min_dimensions(
    wf.lat_opt.outputs.equil_struct, min_dimensions=[7, 7, 7]
)
wf.with_vacancy = create_vacancy(wf.supercell, remove_atom_index=0)
wf.calc_bulk = calculate(
    wf.supercell, engine=engine.with_working_directory("supercell"), label="calc_bulk"
)
wf.calc_vac = calculate(
    wf.with_vacancy, engine=engine.with_working_directory("vacancy"), label="calc_vac"
)
wf.E_form = calculate_vacancy_formation_energy(
    vacancy_energy=wf.calc_vac.outputs.engine_output.final_energy,
    supercell_energy=wf.calc_bulk.outputs.engine_output.final_energy,
    n_atoms_supercell=n_atoms_supercell,
)
wf.run()

a0 = wf.lat_opt.outputs.a0.value
print(f"Optimised bcc Fe lattice parameter (Al-Fe.eam.fs): a0 = {a0:.4f} A (initial scan centre 2.85 A)")

e_f = wf.E_form.outputs.formation_energy.value
print(f"Vacancy formation energy of Fe in BCC Fe (Al-Fe.eam.fs): {e_f:.3f} eV")

supercell: 54 Fe atoms


      Step     Time          Energy          fmax
BFGS:    0 17:58:12       -7.968559        0.000000


      Step     Time          Energy          fmax
BFGS:    0 17:58:12       -8.009283        0.000000
      Step     Time          Energy          fmax
BFGS:    0 17:58:12       -8.025561        0.000000


      Step     Time          Energy          fmax
BFGS:    0 17:58:12       -8.018422        0.000000
      Step     Time          Energy          fmax
BFGS:    0 17:58:12       -7.989335        0.000000


      Step     Time          Energy          fmax
BFGS:    0 17:58:13     -216.701043        0.000000


      Step     Time          Energy          fmax
BFGS:    0 17:58:14     -210.851130        0.351019


BFGS:    1 17:58:15     -210.866653        0.315744


BFGS:    2 17:58:16     -210.939713        0.123929


BFGS:    3 17:58:17     -210.944387        0.112243


BFGS:    4 17:58:18     -210.956916        0.036699
Optimised bcc Fe lattice parameter (Al-Fe.eam.fs): a0 = 2.8554 A (initial scan centre 2.85 A)
Vacancy formation energy of Fe in BCC Fe (Al-Fe.eam.fs): 1.731 eV
